In [ ]:
def f(x: float):
    try:
        print('try')
        r = 10 / x
    except ZeroDivisionError:
        print('except')
        return 'ошибка'
    else:
        print('else')
        return r
    finally:
        print('finally') # никогда не пишите тут return

print(f(2))
print()
print(f(0))

try
else
finally
5.0

try
except
finally
ошибка


In [3]:
def print_tree(cls, indent=0, depth=2):
    print(' ' * indent + cls.__name__)
    if indent < depth:
        for sub_cls in cls.__subclasses__():
            if sub_cls.__module__ == 'builtins':
                print_tree(sub_cls, indent + 1, depth)

print_tree(LookupError)
print()
print_tree(OSError, depth=1)
print()
print(KeyboardInterrupt.__mro__)

LookupError
 IndexError
 KeyError

OSError
 BlockingIOError
 ChildProcessError
 ConnectionError
 FileExistsError
 FileNotFoundError
 InterruptedError
 IsADirectoryError
 NotADirectoryError
 PermissionError
 ProcessLookupError
 TimeoutError

(<class 'KeyboardInterrupt'>, <class 'BaseException'>, <class 'object'>)


In [ ]:
try:
    {}['x']
except LookupError as e:
    print(f'catch KeyError through LookupError:', type(e).__name__, e)

try:
    open('some_file.txt')
except OSError as e:
    print(f'catch FileNotFoundError through OSError:', type(e).__name__, e)

catch KeyError through LookupError: KeyError 'x'
catch FileNotFoundError through OSError: FileNotFoundError [Errno 2] No such file or directory: 'some_file.txt'


In [8]:
try:
    int('abc')
except Exception:
    print('общая ветка')
except ValueError:
    print('скрытая ветка')
# правильно от частоного к общему

общая ветка


In [14]:
# KeyError: 'B-7'
class BankError(Exception):
    """Базовое исключение для всего банковского модуля."""

class AccountNotFound(BankError, LookupError):
    def __init__(self, account_id):
        self.account_id = account_id
        super().__init__(f'счет {account_id} не найден')

class InsufficientcleintError(BankError):
    def __init__(self, account_id, requested, available):
        self.account_id = account_id
        self.requested = requested
        self.available = available
        super().__init__(f'на счете {account_id} недостаточно средств: нужно {requested}, доступно {available}')

    @property
    def shortage(self):
        return self.requested - self.available


class InvalidAmoundError(BankError, ValueError):
    pass


class Bank:
    def __init__(self):
        self._accounts = {}

    def open(self, account_id, balance=0):
        self._accounts[account_id] = balance

    def balance(self, account_id):
        try:
            return self._accounts[account_id]
        except KeyError:
            raise AccountNotFound(account_id=account_id) from None
    
    def withdraw(self, account_id, amount):
        if amount <= 0:
            raise InvalidAmoundError(f'сумма должна быть положительной, получено {amount}')

        available = self.balance(account_id)
        if amount > available:
            raise InsufficientcleintError(account_id, amount, available)

        self._accounts[account_id] -= amount



In [15]:
bank = Bank()
bank.open('A-1', 100)

accounts_l = [('A-1', 30), ('A-1', 500), ('B-7', 10), ('A-1', -10)]

for acc_id, amount in accounts_l:
    try:
        bank.withdraw(acc_id, amount)
        print('ok', 'остаток', bank.balance(acc_id))
    except InsufficientcleintError as e:
        print('не хватает', e.shortage, '>>>>', e)
    except BankError as e:
        print(type(e).__name__, '>>>>', e)

ok остаток 70
не хватает 430 >>>> на счете A-1 недостаточно средств: нужно 500, доступно 70
AccountNotFound >>>> счет B-7 не найден
InvalidAmoundError >>>> сумма должна быть положительной, получено -10


In [18]:
import json
import traceback

class ConfigError(Exception):
    pass

def load_config(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ConfigError('конфиг поврежден') from e

try:
    load_config('{"port": 80,}')
except ConfigError as e:
    print(e)
    print('причина:', repr(e.__cause__))
    # print(traceback.print_exc())

конфиг поврежден
причина: JSONDecodeError('Illegal trailing comma before end of object: line 1 column 12 (char 11)')


raise New() внутри except -> плохая практика
raise New() as e -> причина важна, перевод общей ошибки в свою
raise New() from None -> когда исходная ошибка это деталь реализации и она путает 

In [33]:
# bad pratice
try:
    a = 'abc'
    int(a)
except:
    pass

# good pratice
try:
    a = 'None'
    int(a)
except:
    print(f'данные невалидны {a!r}')

данные невалидны 'None'


In [ ]:
# bp
try:
    ...
    ...
    {}['x']
    ...
    ...
except:
    print('понятный вывод')

# gp
...
...
try:
    {}['x']
except KeyError:
    print('нет такого ключа {a}')
...
...

понятный вывод
нет такого ключа


Ellipsis

In [ ]:
# gp
try:
    data = json.loads('{"port": 80}')
except json.JSONDecodeError as e:
    print('понятная ошибка')
else:
    validate(data)

In [26]:
# bp

def some_f(a, b):
    if b == 0:
        return '-1'
    return a / b

# gp
# лучше зарейзить ошибку, а не вернуть просто ее код

In [ ]:
# лучше использовать подход EAFP, а не LBYL
path = 'просто'
import os
# LBYL
if os.path.exists(path):
    with open(path) as f:
        ...

# EAFP
try:
    f = open(path)
except FileExistsError:
    ...

In [ ]:
import sqlite3


sqlite3.OperationalError # не должна попасть в http модуль
raise ... from e

Домашнее задание

Все файлы кладите в папку `hw11/`. В каждом `.py` файле должен быть блок `if __name__ == '__main__':` с проверками (`assert`), который запускается без ошибок. Приведенные проверки минимальные: добавьте свои, особенно на граничные случаи. При проверке будут использоваться и дополнительные тесты.

Задания 1-3 обязательные. Задание 4 и пункты со звездочкой (*) дополнительные и сложнее.

---

#### Задание 1. Склад: иерархия исключений и транзакции

В файле `hw11/inventory.py` реализуйте класс `Inventory` (учет остатков товаров) и иерархию исключений к нему.

Исключения:
- `InventoryError(Exception)`: базовое для модуля;
- `UnknownItemError`: товара нет на складе. Атрибут `sku`. Должен ловиться и как `InventoryError`, и как `LookupError`;
- `InsufficientStockError`: товара не хватает. Атрибуты `sku`, `requested`, `available` и вычисляемое свойство `shortage`;
- `InvalidQuantityError`: некорректное количество. Ловится и как `InventoryError`, и как `ValueError`.

Методы `Inventory`:
- `add(sku, qty)` добавляет товар (если товара не было, он появляется);
- `remove(sku, qty)` списывает товар;
- `quantity(sku)` возвращает остаток;
- количество должно быть целым положительным числом, иначе `InvalidQuantityError`. Обратите внимание: `True` формально является `int`, но количеством считаться не должно;
- `UnknownItemError` при обращении к неизвестному товару бросается **без** контекста исходного `KeyError`;
- `transaction()` это контекст-менеджер: если внутри блока возникло **любое** исключение (включая `KeyboardInterrupt`), все изменения внутри блока откатываются, а исключение пробрасывается дальше.

```python
if __name__ == '__main__':
    inv = Inventory()
    inv.add('apple', 10)
    inv.add('pear', 3)
    inv.add('apple', 5)
    assert inv.quantity('apple') == 15

    for bad in (0, -1, 2.5, '3', True):
        try:
            inv.add('apple', bad)
            assert False, f'ожидали InvalidQuantityError для {bad!r}'
        except InvalidQuantityError as e:
            assert isinstance(e, ValueError) and isinstance(e, InventoryError)

    try:
        inv.remove('kiwi', 1)
        assert False
    except UnknownItemError as e:
        assert e.sku == 'kiwi' and isinstance(e, LookupError) and 'kiwi' in str(e)
        assert e.__suppress_context__ or e.__context__ is None

    try:
        inv.remove('pear', 5)
        assert False
    except InsufficientStockError as e:
        assert (e.sku, e.requested, e.available, e.shortage) == ('pear', 5, 3, 2)
    assert inv.quantity('pear') == 3

    with inv.transaction():
        inv.remove('apple', 5)
        inv.add('kiwi', 7)
    assert inv.quantity('apple') == 10 and inv.quantity('kiwi') == 7

    try:
        with inv.transaction():
            inv.remove('apple', 4)
            inv.add('plum', 2)
            inv.remove('pear', 100)
        assert False
    except InsufficientStockError:
        pass
    assert inv.quantity('apple') == 10 and inv.quantity('pear') == 3
    try:
        inv.quantity('plum')            # plum добавили внутри отмененной транзакции
        assert False
    except UnknownItemError:
        pass

    try:
        with inv.transaction():
            inv.remove('apple', 1)
            raise KeyboardInterrupt
    except KeyboardInterrupt:
        pass
    assert inv.quantity('apple') == 10
```

В комментарии (2-3 предложения): почему `UnknownItemError` полезно наследовать и от `LookupError`, а `InsufficientStockError` нет?

**1\*.** Поддержите вложенные транзакции: если внутренняя транзакция упала и ошибку поймали внутри внешней, откатываются только изменения внутренней. Если упала внешняя, откатывается все, включая успешно завершенные внутренние.

```python
    with inv.transaction():
        inv.remove('apple', 1)
        try:
            with inv.transaction():
                inv.remove('apple', 2)
                inv.remove('pear', 100)
        except InsufficientStockError:
            pass
        assert inv.quantity('apple') == 9      # откатилось только -2
    assert inv.quantity('apple') == 9

    try:
        with inv.transaction():
            inv.remove('apple', 1)
            with inv.transaction():
                inv.remove('apple', 1)
            raise RuntimeError
    except RuntimeError:
        pass
    assert inv.quantity('apple') == 9          # откатилось все
```

---

#### Задание 2. Загрузчик конфига с понятными ошибками

В файле `hw11/config.py` напишите функцию `load_config(path, schema)`. Она читает файл формата `ключ = значение` (пустые строки и строки с `#` пропускаются) и возвращает словарь, приведенный к типам из `schema`, например `{'host': str, 'port': int, 'debug': bool, 'timeout': float}`. Ключи, которых нет в схеме, игнорируются.

Для `bool` допустимы значения `true/false`, `yes/no`, `1/0` в любом регистре. Обратите внимание, что `bool('false')` в Python равно `True`.

Исключения:
- `ConfigError(Exception)`: базовое;
- `ConfigNotFoundError`: файла нет. Ловится и как `ConfigError`, и как `FileNotFoundError`. Исходная ошибка ОС сохраняется в `__cause__`. Сообщение содержит путь;
- `ConfigSyntaxError`: строка без `=` или с пустым ключом. Атрибуты `lineno` (номер строки в файле) и `line` (текст строки без пробелов по краям);
- `MissingKeyError`: в файле нет ключа из схемы. Атрибут `key`;
- `InvalidValueError`: значение не приводится к типу. Ловится и как `ConfigError`, и как `ValueError`. Атрибуты `key`, `value` (исходная строка), `expected` (тип). Если приведение упало с исключением, оно сохраняется в `__cause__`.

Функция бросает **первую** найденную ошибку: сначала проверяется синтаксис всего файла, затем ключи схемы в порядке их объявления.

```python
if __name__ == '__main__':
    SCHEMA = {'host': str, 'port': int, 'debug': bool, 'timeout': float}

    with open('good.cfg', 'w', encoding='utf-8') as f:
        f.write('# сервер\nhost = localhost\nport = 8080\n\ndebug = yes\ntimeout = 2.5\nextra = ignored\n')
    assert load_config('good.cfg', SCHEMA) == {
        'host': 'localhost', 'port': 8080, 'debug': True, 'timeout': 2.5}

    try:
        load_config('nope.cfg', SCHEMA)
        assert False
    except ConfigNotFoundError as e:
        assert isinstance(e, FileNotFoundError) and isinstance(e, ConfigError)
        assert isinstance(e.__cause__, FileNotFoundError) and 'nope.cfg' in str(e)

    with open('bad.cfg', 'w', encoding='utf-8') as f:
        f.write('host = localhost\nport = eighty\nэто не строка конфига\ndebug = maybe\n')
    try:
        load_config('bad.cfg', SCHEMA)
        assert False
    except ConfigSyntaxError as e:
        assert (e.lineno, e.line) == (3, 'это не строка конфига')

    with open('bad2.cfg', 'w', encoding='utf-8') as f:
        f.write('host = localhost\nport = eighty\ndebug = no\n')
    try:
        load_config('bad2.cfg', SCHEMA)
        assert False
    except InvalidValueError as e:
        assert (e.key, e.value, e.expected) == ('port', 'eighty', int)
        assert isinstance(e.__cause__, ValueError) and isinstance(e, ValueError)

    with open('bad3.cfg', 'w', encoding='utf-8') as f:
        f.write('host = localhost\nport = 80\ndebug = no\n')
    try:
        load_config('bad3.cfg', SCHEMA)
        assert False
    except MissingKeyError as e:
        assert e.key == 'timeout'
```

В комментарии ответьте:
1. Почему для `ConfigNotFoundError` нужен `from e`, а не `from None`?
2. Если унаследовать `ConfigNotFoundError` от `FileNotFoundError`, `str(e)` может выглядеть неожиданно. Проверьте, что выводит `str(e)` при наследовании от `OSError`, и объясните, как вы добились понятного сообщения.

**2\*.** Добавьте параметр `strict=True`. При `strict=False` функция не останавливается на первой ошибке, а собирает **все** и бросает их одним `ExceptionGroup`. Вызывающий код обрабатывает их через `except*`:

```python
    caught = []
    try:
        load_config('bad.cfg', SCHEMA, strict=False)
    except* ConfigSyntaxError as eg:
        caught.append(('syntax', [e.lineno for e in eg.exceptions]))
    except* InvalidValueError as eg:
        caught.append(('value', sorted(e.key for e in eg.exceptions)))
    except* MissingKeyError as eg:
        caught.append(('missing', [e.key for e in eg.exceptions]))
    assert sorted(caught) == [('missing', ['timeout']), ('syntax', [3]), ('value', ['debug', 'port'])]
```

---

#### Задание 3. Временная подмена атрибутов

В файле `hw11/temp_attrs.py` напишите контекст-менеджер `temporary_attrs(obj, **changes)`. Он на время блока `with` устанавливает объекту атрибуты, а после блока **в любом случае**, даже при исключении, возвращает все как было. Так в тестах временно подменяют настройки или функции (позже вы увидите то же самое в `unittest.mock.patch`).

Требования:
- атрибуты, которых у объекта не было, после блока удаляются;
- блок возвращает сам объект (`as obj`);
- если один из атрибутов установить не удалось (например, из-за `__slots__`), уже установленные откатываются, а исключение пробрасывается;
- **главная тонкость, вспомните лекцию 4:** если атрибут жил в **классе**, а не в экземпляре, то после блока он снова должен браться из класса. Если просто сделать `setattr(obj, name, старое_значение)`, у экземпляра появится собственный атрибут, который затенит атрибут класса. Проверка `'debug' not in vars(s)` это ловит.

```python
if __name__ == '__main__':
    class Settings:
        debug = False                  # атрибут класса

        def __init__(self):
            self.timeout = 10          # атрибут экземпляра

    s = Settings()
    with temporary_attrs(s, timeout=1, debug=True, region='eu') as obj:
        assert obj is s and (s.timeout, s.debug, s.region) == (1, True, 'eu')
    assert s.timeout == 10 and s.debug is False and not hasattr(s, 'region')
    assert 'debug' not in vars(s)       # атрибут снова берется из класса

    try:
        with temporary_attrs(s, timeout=0):
            raise RuntimeError('упали')
    except RuntimeError:
        pass
    assert s.timeout == 10

    class Slotted:
        __slots__ = ('a',)
        def __init__(self):
            self.a = 1

    sl = Slotted()
    try:
        with temporary_attrs(sl, a=2, b=3):     # b нельзя создать
            assert False, 'тело не должно выполниться'
    except AttributeError:
        pass
    assert sl.a == 1

    import math
    with temporary_attrs(math, pi=3):
        assert math.pi == 3
    assert math.pi != 3
```

Можно писать через класс или через `@contextmanager`. В комментарии объясните, почему внутри обязателен `try`/`finally`.

---

#### Задание 4\*. Сборщик ошибок для пакетной обработки

При обработке большого пакета данных (строки файла, письма, платежи) обычно нельзя останавливаться на первой ошибке. Нужно обработать все, что получается, а в конце сообщить обо всех ошибках сразу. В файле `hw11/collector.py` напишите класс `ErrorCollector(*exc_types)`:
- `collector.catch(item)` это контекст-менеджер. Он перехватывает исключения указанных типов (по умолчанию `Exception`), добавляет к исключению заметку `при обработке <repr(item)>` через `add_note` и сохраняет пару `(item, исключение)` в `collector.errors`. Исключения **других** типов пролетают насквозь;
- `bool(collector)` показывает, были ли ошибки;
- `raise_if_any(message=...)` бросает `ExceptionGroup` со всеми собранными исключениями, а если ошибок не было, ничего не делает.

```python
if __name__ == '__main__':
    def process(x):
        if x < 0:
            raise ValueError(f'отрицательное число {x}')
        if x == 0:
            raise ZeroDivisionError('деление на ноль')
        return 10 / x

    collector = ErrorCollector(ValueError, ZeroDivisionError)
    results = []
    for x in [5, -1, 2, 0, -7]:
        with collector.catch(x):
            results.append(process(x))

    assert results == [2.0, 5.0] and bool(collector)
    assert [item for item, _ in collector.errors] == [-1, 0, -7]
    assert collector.errors[0][1].__notes__ == ['при обработке -1']

    try:
        with ErrorCollector(ValueError).catch('x'):
            raise TypeError('не наш тип')
        assert False
    except TypeError:
        pass

    got = {}
    try:
        collector.raise_if_any()
    except* ValueError as eg:
        got['value'] = len(eg.exceptions)
    except* ZeroDivisionError as eg:
        got['zero'] = len(eg.exceptions)
    assert got == {'value': 2, 'zero': 1}

    ErrorCollector().raise_if_any()          # ошибок нет, ничего не происходит
```

Используйте `ErrorCollector`, чтобы переписать `read_sales` из ДЗ к лекциям 6-7: битые строки собираются в коллектор, а не в список `errors`. В комментарии сравните два подхода.